# BloodMNIST: Convolutional neural network
Cleaned course-project workflow. Historical outputs are preserved separately in `archive/` and `results/reference/`.
Run all cells in order. No new training results are embedded in this notebook.


## Data preparation
RGB inputs, fixed seed, and an unweighted cross-entropy baseline matching the original training cell.

In [ ]:
import random
import numpy as np
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from medmnist import INFO, BloodMNIST

SEED = 123
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(2)
device = torch.device("cpu")
info = INFO["bloodmnist"]
data_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])
train_dataset = BloodMNIST(split="train", transform=data_transform, download=True)
test_dataset = BloodMNIST(split="test", transform=data_transform, download=True)
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)


### Visual Inspection of BloodMNIST Samples

In [ ]:
import matplotlib.pyplot as plt
import torchvision

# Get 100 images and labels
images, labels = next(iter(DataLoader(dataset=train_dataset, batch_size=100, shuffle=True)))

# Create a grid of images
grid_img = torchvision.utils.make_grid(images, nrow=10, padding=2, normalize=True)

# Plot the image grid
plt.figure(figsize=(12, 12))
plt.imshow(grid_img.permute(1, 2, 0))  # Convert from (C, H, W) to (H, W, C)
plt.axis('off')
plt.title('100 BloodMNIST Sample Images')
plt.show()


### Dataset Structure Summary

In [ ]:
# Extract details
sample_image, sample_label = train_dataset[0]
train_size = len(train_dataset)
test_size = len(test_dataset)
image_shape = sample_image.shape
label_names = info['label']

print("Image shape (C, H, W):", image_shape)
print("Sample label index:", sample_label)
print("Label names:", label_names)
print("Training set size:", train_size)
print("Test set size:", test_size)

### Class Distribution in Training Set

This section computes and displays the class prevalence in the BloodMNIST training dataset.

In [ ]:
import pandas as pd
import numpy as np

# Extract all labels from the training dataset
all_labels = np.array([label for _, label in train_dataset])

# BloodMNIST labels are integer class IDs in shape (N, 1).
all_labels = all_labels.reshape(-1)

# Count occurrences of each label
unique, counts = np.unique(all_labels, return_counts=True)

# Create prevalence table
label_names = info['label']
label_table = pd.DataFrame({
    'Label Index': unique,
    'Label Name': [label_names[str(i)] for i in unique],
    'Count': counts,
    'Percentage': (counts / counts.sum() * 100).round(2)
})

print(label_table)


### Convolutional Neural Network

This workflow uses a custom convolutional neural network (CNN) in PyTorch for multi-class classification of blood cell images.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# Define the two-block CNN baseline
class BloodMNIST_CNN(nn.Module):
    def __init__(self, in_channels=3, num_classes=8):
        super(BloodMNIST_CNN, self).__init__()

        # First convolutional layer: 3 input channels → 32 output channels
        self.conv1 = nn.Conv2d(in_channels, 32, kernel_size=3, padding=1)
        # Second convolutional layer: 32 → 64 output channels
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)

        # Max pooling to reduce spatial dimensions by factor of 2
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        # Fully connected layers
        # After 2 poolings, 28x28 → 14x14 → 7x7 → 64 channels
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        # Apply convolution → ReLU → pooling
        x = self.pool(F.relu(self.conv1(x)))  # shape: (32, 14, 14)
        x = self.pool(F.relu(self.conv2(x)))  # shape: (64, 7, 7)

        # Flatten the feature map into a vector
        x = x.view(x.size(0), -1)  # shape: (batch_size, 64*7*7)

        # Fully connected layers
        x = F.relu(self.fc1(x))
        x = self.fc2(x)

        return x

# Instantiate the model
cnn_model = BloodMNIST_CNN(in_channels=3, num_classes=8)

### Optimizer

The workflow uses the Adam optimizer from PyTorch to update the model parameters during training

In [ ]:
import torch.optim as optim

optimizer = optim.Adam(cnn_model.parameters(), lr=0.001)

### Model Training

The model is trained for 10 epochs using the Adam optimizer and cross-entropy loss:


In [ ]:
import torch
import torch.nn as nn

# Move model to CPU (or GPU if available and working)
device = torch.device("cpu")  # Force CPU to avoid libnccl errors
cnn_model.to(device)

# Define loss function: CrossEntropyLoss for multi-class classification
criterion = nn.CrossEntropyLoss()

# Train the model for 10 epochs
for epoch in range(10):
    cnn_model.train()  # Set model to training mode
    running_loss = 0.0

    # Iterate over all mini-batches in the training set
    for images, labels in train_loader:
        # Move inputs and labels to the same device as model
        images = images.to(device)
        labels = labels.reshape(-1).long().to(device)

        # Reset gradient accumulators
        optimizer.zero_grad()

        # Forward pass: compute model predictions
        outputs = cnn_model(images)

        # Compute loss between predictions and actual labels
        loss = criterion(outputs, labels)

        # Backward pass: compute gradient of loss w.r.t. model parameters
        loss.backward()

        # Update model weights using optimizer
        optimizer.step()

        # Accumulate total loss for this mini-batch
        running_loss += loss.item() * images.size(0)

    # Compute and print average loss for this epoch
    epoch_loss = running_loss / len(train_loader.dataset)
    print(f"Epoch {epoch+1}/10 - Loss: {epoch_loss:.4f}")


### Model Evaluation

After training, we evaluate the model's performance on the test set:


In [ ]:
import torch
from sklearn.metrics import roc_auc_score, accuracy_score
import numpy as np
from sklearn.metrics import classification_report

# Set model to evaluation mode
cnn_model.eval()

# Ensure model runs on CPU (or GPU if using it)
device = torch.device("cpu")
cnn_model.to(device)

# Prepare containers to store predictions and true labels
all_probs = []   # for predicted probabilities
all_preds = []   # for predicted class labels
all_targets = [] # for true labels

# Disable gradient computation for evaluation (faster, less memory)
with torch.no_grad():
    for images, labels in test_loader:
        # Move data to same device as model
        images = images.to(device)
        labels = labels.reshape(-1).long().to(device)

        # Get raw model outputs (logits)
        outputs = cnn_model(images)

        # Apply softmax to convert logits to predicted class probabilities
        probs = torch.softmax(outputs, dim=1)

        # Get predicted class labels (index of max probability)
        preds = torch.argmax(probs, dim=1)

        # Store results for evaluation
        all_probs.append(probs.cpu().numpy())     # shape: (batch_size, num_classes)
        all_preds.append(preds.cpu().numpy())     # shape: (batch_size,)
        all_targets.append(labels.cpu().numpy())  # shape: (batch_size,)

# Concatenate results from all batches into full arrays
all_probs = np.concatenate(all_probs, axis=0)
all_preds = np.concatenate(all_preds, axis=0)
all_targets = np.concatenate(all_targets, axis=0)

# Compute overall classification accuracy
accuracy = accuracy_score(all_targets, all_preds)

# Convert labels to one-hot encoding for AUC computation
num_classes = all_probs.shape[1]
targets_onehot = np.eye(num_classes)[all_targets]

# Compute macro-average one-vs-rest AUC
auc = roc_auc_score(targets_onehot, all_probs, average='macro', multi_class='ovr')

# Print metrics
print(f"Test Accuracy: {accuracy:.4f}")
print(f"One-vs-Rest AUC (macro-average): {auc:.4f}")



# Compute AUC for each class (one-vs-rest)
print("\nAUC for each class:")
for i in range(num_classes):
    class_auc = roc_auc_score(targets_onehot[:, i], all_probs[:, i])
    print(f"Class {i}: AUC = {class_auc:.4f}")

# Print precision, recall, F1-score for each class
print("\nClassification Report:")
print(classification_report(all_targets, all_preds, digits=4))


In [ ]:
from pathlib import Path
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, classification_report

out = Path("results/rerun")
out.mkdir(parents=True, exist_ok=True)
pd.DataFrame([{"model": "cnn",
    "accuracy": accuracy_score(all_targets, all_preds),
    "macro_f1": f1_score(all_targets, all_preds, average="macro"),
    "macro_auc": auc, "test_images": len(all_targets)
}]).to_csv(out / "cnn_metrics.csv", index=False)
pd.DataFrame(classification_report(all_targets, all_preds, output_dict=True)).T.to_csv(
    out / "cnn_class_metrics.csv")
print("New aggregate results saved to", out)


## Interpretation
Compare results with the archived tables, but expect differences across runs and package versions. The CNN seed was added during cleanup. Do not select hyperparameters using the test set; use the official validation split for further development. No external clinical validation is claimed.
